[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vkoul/advanced-ab-testing/blob/main/study_notes/week06_ratio_metrics.ipynb)

> **Run this notebook interactively:** Click the badge above to open in Google Colab.

# Week 6: Ratio Metrics & Linearization

**Course**: Advanced A/B Testing | **Context**: QuickCart (online electronics retailer)

**Your role**: Data scientist on the experimentation platform team. QuickCart's key business metrics are ratios — revenue per session, conversion rate per visit, average order value. These can't be naively analyzed with a t-test. This week you'll learn why, and master the industry-standard solution: linearization.

---

## What You'll Learn

1. Why ratio metrics break standard testing assumptions
2. Three approaches: user-average, delta method, and linearization
3. The linearization formula and why it works
4. Full implementation with calibration validation
5. Combining linearization with CUPED
6. The bucket method for temporal dependencies
7. QuickCart example: revenue per search session

---

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

from collections import defaultdict
from tqdm.notebook import tqdm

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
np.random.seed(42)

---

## 1. The Problem with Ratio Metrics

### What is a Ratio Metric?

A ratio metric takes the form:

$$
R = \frac{\sum_i x_i}{\sum_i y_i} = \frac{\text{total numerator}}{\text{total denominator}}
$$

Examples at QuickCart:
- **Revenue per session** = total revenue / total sessions
- **Conversion rate** = total purchases / total visits
- **Average order value** = total revenue / total orders
- **Click-through rate** = total clicks / total impressions

### Why Standard Approaches Fail

The fundamental issue: **users contribute different numbers of observations**.

- User A has 2 sessions with revenue [0, 100] → revenue/session = 50
- User B has 50 sessions with revenue [0, 0, ..., 0, 5] → revenue/session = 0.1

If we naively compute per-user averages and run a t-test, we give equal weight to User A and User B. But User B contributes 50 sessions while User A contributes only 2. The global ratio should be dominated by User B's experience.

This mismatch between user-level averaging and the global ratio creates **bias** in the test.

In [ ]:
# Demonstrate the problem
np.random.seed(42)

# Simulate users with varying session counts
n_users = 1000
# Session counts follow a heavy-tailed distribution (realistic)
sessions_per_user = np.random.negative_binomial(n=2, p=0.1, size=n_users) + 1

print(f"Sessions per user statistics:")
print(f"  Min: {sessions_per_user.min()}, Max: {sessions_per_user.max()}")
print(f"  Mean: {sessions_per_user.mean():.1f}, Median: {np.median(sessions_per_user):.0f}")
print(f"  Total sessions: {sessions_per_user.sum():,}")

# Revenue per session: each session has some probability of generating revenue
revenue_per_session = 5.0  # true average

# Generate session-level data
all_revenues = []
user_ids = []
for i, n_sess in enumerate(sessions_per_user):
    # Revenue per session ~ Exponential (most sessions = $0, some have purchases)
    has_purchase = np.random.binomial(1, 0.3, n_sess)
    session_revenue = has_purchase * np.random.exponential(revenue_per_session / 0.3, n_sess)
    all_revenues.extend(session_revenue)
    user_ids.extend([i] * n_sess)

df_sessions = pd.DataFrame({'user_id': user_ids, 'revenue': all_revenues})

# Compute the ratio metric two ways
global_ratio = df_sessions['revenue'].sum() / len(df_sessions)
user_averages = df_sessions.groupby('user_id')['revenue'].mean()
user_avg_of_avgs = user_averages.mean()

print(f"\nGlobal ratio (sum revenue / total sessions): ${global_ratio:.2f}")
print(f"Average of user-level averages:               ${user_avg_of_avgs:.2f}")
print(f"Difference: ${user_avg_of_avgs - global_ratio:.2f} ({(user_avg_of_avgs/global_ratio - 1)*100:.1f}% bias)")
print(f"\nThese differ because users with few sessions (and potentially extreme")
print(f"per-session averages) get overweighted in the user-average approach.")

---

## 2. Three Approaches to Testing Ratio Metrics

### Approach 1: User-Average Method

Compute $r_i = x_i / y_i$ for each user, then run a t-test on these user-level ratios.

**Pros**: Simple, gives one observation per user (independent).

**Cons**: Biased when session counts vary across users. Users with 1 session dominate the variance.

### Approach 2: Delta Method

Analytically estimate the variance of the ratio using a Taylor expansion:

$$
\text{Var}\left(\frac{\bar{X}}{\bar{Y}}\right) \approx \frac{1}{n} \left( \frac{\sigma_X^2}{\mu_Y^2} - 2\frac{\text{Cov}(X,Y) \cdot \mu_X}{\mu_Y^3} + \frac{\mu_X^2 \sigma_Y^2}{\mu_Y^4} \right)
$$

**Pros**: Asymptotically correct, accounts for correlation between numerator and denominator.

**Cons**: Complex formula, relies on large-sample approximation.

### Approach 3: Linearization (Recommended)

Transform the ratio metric into a linear (per-user sum) metric that can be tested with a standard t-test:

$$
L_i = x_i - \kappa \cdot y_i
$$

where $\kappa = \frac{\sum_i x_i}{\sum_i y_i}$ is the global ratio (estimated from data).

**Pros**: Simple, correctly calibrated, compatible with CUPED.

**Cons**: Requires estimating $\kappa$ (use control group or pooled data).

:::{admonition} Why linearization works — Taylor expansion derivation (click to expand)
:class: dropdown

We want to test whether the ratio $R = \bar{X}/\bar{Y}$ differs between treatment and control.

Define $f(\bar{X}, \bar{Y}) = \bar{X}/\bar{Y}$. By first-order Taylor expansion around $(\mu_X, \mu_Y)$:

$$
R \approx \frac{\mu_X}{\mu_Y} + \frac{1}{\mu_Y}(\bar{X} - \mu_X) - \frac{\mu_X}{\mu_Y^2}(\bar{Y} - \mu_Y)
$$

Define $\kappa = \mu_X / \mu_Y$ (the true global ratio). Then:

$$
R \approx \kappa + \frac{1}{\mu_Y}(\bar{X} - \kappa \bar{Y})
$$

The term $\bar{X} - \kappa \bar{Y} = \frac{1}{n}\sum_i (x_i - \kappa y_i)$ is a simple average of per-user quantities!

Define the **linearized metric** for user $i$:

$$
L_i = x_i - \kappa \cdot y_i
$$

Testing whether $R^{\text{treat}} \neq R^{\text{ctrl}}$ is equivalent to testing whether $\bar{L}^{\text{treat}} \neq \bar{L}^{\text{ctrl}}$.

The key insight: $L_i$ is a **per-user sum**, so observations are independent across users. We can apply a standard t-test directly.

In practice, we estimate $\kappa$ as $\sum x_i / \sum y_i$ from the control group (or pooled data). This is equivalent to the delta method but operationally simpler.
:::

---

## 3. Implementation: `calculate_linearized_metric`

In [ ]:
def calculate_linearized_metric(
    df: pd.DataFrame,
    value_name: str,
    user_id_name: str,
    list_user_id: list,
    date_name: str,
    period: tuple,
    metric_name: str,
    kappa: float = None
) -> pd.DataFrame:
    """
    Calculate the linearized ratio metric for a set of users.
    
    The ratio metric is: sum(value) / count(observations)
    Linearized version: sum_per_user - kappa * count_per_user
    
    Parameters
    ----------
    df : DataFrame with session-level data
    value_name : name of the numerator column (e.g., 'revenue')
    user_id_name : name of the user identifier column
    list_user_id : list of user IDs to include
    date_name : name of the date column
    period : tuple (start_date, end_date) for the analysis period
    metric_name : name for the output column
    kappa : global ratio (sum_values / count_observations). 
            If None, computed from the provided data.
    
    Returns
    -------
    DataFrame with columns [user_id_name, metric_name] containing
    the linearized metric per user, plus columns for the components.
    """
    # Filter to target users and period
    df_filtered = df[df[user_id_name].isin(list_user_id)].copy()
    df_filtered[date_name] = pd.to_datetime(df_filtered[date_name])
    
    period_start, period_end = pd.to_datetime(period[0]), pd.to_datetime(period[1])
    mask = (df_filtered[date_name] >= period_start) & (df_filtered[date_name] <= period_end)
    df_period = df_filtered[mask]
    
    # Step 1: Compute sum and count per user
    user_stats = (
        df_period
        .groupby(user_id_name)
        .agg(
            value_sum=(value_name, 'sum'),
            session_count=(value_name, 'count')
        )
        .reindex(list_user_id, fill_value=0)
    )
    
    # Step 2: Compute kappa if not provided
    if kappa is None:
        total_value = user_stats['value_sum'].sum()
        total_count = user_stats['session_count'].sum()
        kappa = total_value / total_count if total_count > 0 else 0
    
    # Step 3: Compute linearized metric
    linearized = user_stats['value_sum'] - kappa * user_stats['session_count']
    
    result = pd.DataFrame({
        user_id_name: list_user_id,
        metric_name: linearized.values,
        'value_sum': user_stats['value_sum'].values,
        'session_count': user_stats['session_count'].values
    })
    
    return result, kappa

---

## 4. A/A Test Simulation: Validating Calibration

The most important validation: under the null hypothesis (no treatment effect), p-values should be uniform. Let's compare all three approaches.

In [ ]:
def generate_session_data(n_users, sessions_distribution='negative_binomial',
                          revenue_per_session=5.0, treatment_effect=0.0,
                          treatment_mask=None):
    """
    Generate synthetic session-level data with varying sessions per user.
    
    Returns DataFrame with columns: user_id, revenue, session_count
    and the per-user aggregated stats.
    """
    # Session counts: heavy-tailed distribution
    if sessions_distribution == 'negative_binomial':
        sessions_per_user = np.random.negative_binomial(n=2, p=0.15, size=n_users) + 1
    else:
        sessions_per_user = np.random.poisson(lam=10, size=n_users) + 1
    
    # Generate session-level revenue
    user_sums = np.zeros(n_users)
    for i in range(n_users):
        n_sess = sessions_per_user[i]
        base_revenue = np.random.exponential(scale=revenue_per_session, size=n_sess)
        # Apply treatment effect (multiplicative)
        if treatment_mask is not None and treatment_mask[i]:
            base_revenue *= (1 + treatment_effect)
        user_sums[i] = base_revenue.sum()
    
    return user_sums, sessions_per_user


def run_aa_simulation(n_simulations=5000, n_users=2000):
    """
    Run A/A test simulation comparing three approaches for ratio metrics.
    """
    pvals_user_avg = []
    pvals_linearized = []
    pvals_naive_session = []  # treating sessions as independent
    
    n_half = n_users // 2
    
    for _ in range(n_simulations):
        # Generate A/A data (no effect)
        user_sums, session_counts = generate_session_data(
            n_users, treatment_effect=0.0
        )
        
        # Split into two groups
        sums_a, sums_b = user_sums[:n_half], user_sums[n_half:]
        counts_a, counts_b = session_counts[:n_half], session_counts[n_half:]
        
        # Method 1: User-average (ratio per user)
        ratio_a = sums_a / counts_a
        ratio_b = sums_b / counts_b
        _, p_user_avg = stats.ttest_ind(ratio_a, ratio_b)
        pvals_user_avg.append(p_user_avg)
        
        # Method 2: Linearization
        # Compute kappa from pooled data
        kappa = user_sums.sum() / session_counts.sum()
        linear_a = sums_a - kappa * counts_a
        linear_b = sums_b - kappa * counts_b
        _, p_linear = stats.ttest_ind(linear_a, linear_b)
        pvals_linearized.append(p_linear)
        
        # Method 3: Naive (ignore user structure, sum per user, t-test on sums)
        # This is wrong because users with more sessions have larger sums
        _, p_naive = stats.ttest_ind(sums_a, sums_b)
        pvals_naive_session.append(p_naive)
    
    return pvals_user_avg, pvals_linearized, pvals_naive_session


print("Running A/A simulation (5000 iterations)...")
pvals_ua, pvals_lin, pvals_naive = run_aa_simulation(n_simulations=5000)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

methods = [
    ('User-Average', pvals_ua, 'steelblue'),
    ('Linearization', pvals_lin, 'darkorange'),
    ('Naive (sum per user)', pvals_naive, 'green'),
]

for ax, (name, pvals, color) in zip(axes, methods):
    fpr = np.mean(np.array(pvals) < 0.05)
    ax.hist(pvals, bins=50, density=True, alpha=0.7, color=color)
    ax.axhline(1, color='red', linestyle='--', linewidth=2, label='Uniform (ideal)')
    ax.set_title(f'{name}\nFPR = {fpr:.3f} (target: 0.050)')
    ax.set_xlabel('p-value')
    ax.set_ylabel('Density')
    ax.set_ylim([0, 2.0])
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\nFalse Positive Rates (target = 0.050):")
print(f"  User-Average:       {np.mean(np.array(pvals_ua) < 0.05):.3f}")
print(f"  Linearization:      {np.mean(np.array(pvals_lin) < 0.05):.3f}")
print(f"  Naive (sum/user):   {np.mean(np.array(pvals_naive) < 0.05):.3f}")

### Interpretation

- **User-Average**: May show slight miscalibration because it gives equal weight to all users regardless of how many sessions they have. Users with 1 session have extremely noisy ratio estimates.
- **Linearization**: Correctly calibrated (FPR close to 0.05, uniform p-values). This is the recommended approach.
- **Naive sum**: Also approximately calibrated under A/A (since means are equal), but tests the wrong hypothesis — it's sensitive to differences in total spending, not in the ratio metric.

---

## 5. Power Comparison: Which Method Detects Real Effects?

Now let's add a true treatment effect and compare detection rates.

In [ ]:
def run_power_simulation(treatment_effect, n_simulations=3000, n_users=2000):
    """Run simulation with a specified treatment effect on the ratio metric."""
    pvals_user_avg = []
    pvals_linearized = []
    
    n_half = n_users // 2
    treatment_mask = np.zeros(n_users, dtype=bool)
    treatment_mask[n_half:] = True
    
    for _ in range(n_simulations):
        user_sums, session_counts = generate_session_data(
            n_users, treatment_effect=treatment_effect,
            treatment_mask=treatment_mask
        )
        
        sums_a, sums_b = user_sums[:n_half], user_sums[n_half:]
        counts_a, counts_b = session_counts[:n_half], session_counts[n_half:]
        
        # User-average
        ratio_a = sums_a / counts_a
        ratio_b = sums_b / counts_b
        _, p_ua = stats.ttest_ind(ratio_b, ratio_a)
        pvals_user_avg.append(p_ua)
        
        # Linearization (kappa from control)
        kappa = sums_a.sum() / counts_a.sum()
        linear_a = sums_a - kappa * counts_a
        linear_b = sums_b - kappa * counts_b
        _, p_lin = stats.ttest_ind(linear_b, linear_a)
        pvals_linearized.append(p_lin)
    
    power_ua = np.mean(np.array(pvals_user_avg) < 0.05)
    power_lin = np.mean(np.array(pvals_linearized) < 0.05)
    return power_ua, power_lin


# Test across different effect sizes
effects = [0, 0.02, 0.05, 0.08, 0.10, 0.15, 0.20]
powers_ua = []
powers_lin = []

print("Running power simulation across effect sizes...")
for effect in tqdm(effects):
    p_ua, p_lin = run_power_simulation(effect, n_simulations=2000)
    powers_ua.append(p_ua)
    powers_lin.append(p_lin)

plt.plot(effects, powers_ua, 'o-', label='User-Average', linewidth=2)
plt.plot(effects, powers_lin, 's-', label='Linearization', linewidth=2)
plt.axhline(0.05, color='gray', linestyle=':', label='alpha = 0.05 (no effect)')
plt.axhline(0.8, color='red', linestyle='--', alpha=0.5, label='80% power target')
plt.xlabel('Treatment Effect (multiplicative lift)')
plt.ylabel('Power')
plt.title('Power of Ratio Metric Tests')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print("\nPower at 10% effect:")
idx_10 = effects.index(0.10)
print(f"  User-Average:  {powers_ua[idx_10]:.3f}")
print(f"  Linearization: {powers_lin[idx_10]:.3f}")

---

## 6. QuickCart Example: Revenue per Search Session

QuickCart has deployed a new search ranking algorithm. The primary metric is **revenue per search session** — we want to know if users who search generate more revenue per search under the new algorithm.

This is a classic ratio metric: total revenue from search / total search sessions.

In [ ]:
np.random.seed(42)
n_users = 5000
n_days = 14  # 2-week experiment

user_ids = [f"user_{i:05d}" for i in range(n_users)]
control_users = user_ids[:n_users // 2]
treatment_users = user_ids[n_users // 2:]

# User characteristics
user_search_propensity = np.random.lognormal(mean=0.5, sigma=0.8, size=n_users)
user_purchase_propensity = np.random.beta(2, 8, size=n_users)  # conversion rate

# Generate search sessions
records = []
base_date = pd.Timestamp('2024-03-01')

for i, uid in enumerate(user_ids):
    is_treatment = uid in treatment_users
    
    for day in range(n_days):
        # Number of search sessions this day
        n_searches = np.random.poisson(lam=user_search_propensity[i])
        
        for _ in range(n_searches):
            # Does this search lead to a purchase?
            base_conv_rate = user_purchase_propensity[i]
            # Treatment increases conversion by 8%
            if is_treatment:
                conv_rate = base_conv_rate * 1.08
            else:
                conv_rate = base_conv_rate
            
            purchased = np.random.binomial(1, min(conv_rate, 1.0))
            if purchased:
                revenue = np.random.lognormal(mean=3.5, sigma=1.0)
            else:
                revenue = 0.0
            
            records.append({
                'user_id': uid,
                'date': base_date + pd.Timedelta(days=day),
                'search_revenue': round(revenue, 2)
            })

df_search = pd.DataFrame(records)
print(f"Generated {len(df_search):,} search sessions for {n_users} users")
print(f"\nGlobal revenue per search session:")
print(f"  Control:   ${df_search[df_search['user_id'].isin(control_users)]['search_revenue'].sum() / df_search[df_search['user_id'].isin(control_users)].shape[0]:.3f}")
print(f"  Treatment: ${df_search[df_search['user_id'].isin(treatment_users)]['search_revenue'].sum() / df_search[df_search['user_id'].isin(treatment_users)].shape[0]:.3f}")

# Show session count distribution
session_counts = df_search.groupby('user_id').size()
print(f"\nSessions per user: mean={session_counts.mean():.1f}, median={session_counts.median():.0f}, max={session_counts.max()}")

In [ ]:
# Test with all three methods
period = ('2024-03-01', '2024-03-14')

# --- Method 1: User-Average ---
user_metrics = df_search.groupby('user_id').agg(
    total_revenue=('search_revenue', 'sum'),
    n_sessions=('search_revenue', 'count')
)
user_metrics['revenue_per_session'] = user_metrics['total_revenue'] / user_metrics['n_sessions']

ctrl_avg = user_metrics.loc[user_metrics.index.isin(control_users), 'revenue_per_session']
treat_avg = user_metrics.loc[user_metrics.index.isin(treatment_users), 'revenue_per_session']
_, p_user_avg = stats.ttest_ind(treat_avg.dropna(), ctrl_avg.dropna())

# --- Method 2: Linearization ---
# Compute kappa from control group
ctrl_data = df_search[df_search['user_id'].isin(control_users)]
kappa = ctrl_data['search_revenue'].sum() / len(ctrl_data)

linearized_ctrl, _ = calculate_linearized_metric(
    df=df_search, value_name='search_revenue', user_id_name='user_id',
    list_user_id=control_users, date_name='date', period=period,
    metric_name='linearized_rps', kappa=kappa
)

linearized_treat, _ = calculate_linearized_metric(
    df=df_search, value_name='search_revenue', user_id_name='user_id',
    list_user_id=treatment_users, date_name='date', period=period,
    metric_name='linearized_rps', kappa=kappa
)

_, p_linearized = stats.ttest_ind(
    linearized_treat['linearized_rps'].values,
    linearized_ctrl['linearized_rps'].values
)

# --- Method 3: Naive sum per user ---
ctrl_sums = user_metrics.loc[user_metrics.index.isin(control_users), 'total_revenue']
treat_sums = user_metrics.loc[user_metrics.index.isin(treatment_users), 'total_revenue']
_, p_naive = stats.ttest_ind(treat_sums, ctrl_sums)

# Report
print("=" * 65)
print("QuickCart Search Experiment: Revenue per Search Session")
print("=" * 65)
print(f"\nGlobal kappa (control revenue/session): ${kappa:.4f}")
print(f"\n{'Method':<30} {'p-value':>10} {'Significant?':>12}")
print("-" * 55)
print(f"{'User-Average (ratio/user)':<30} {p_user_avg:>10.4f} {'Yes' if p_user_avg < 0.05 else 'No':>12}")
print(f"{'Linearization':<30} {p_linearized:>10.4f} {'Yes' if p_linearized < 0.05 else 'No':>12}")
print(f"{'Naive sum per user':<30} {p_naive:>10.4f} {'Yes' if p_naive < 0.05 else 'No':>12}")

---

## 7. Combining Linearization with CUPED

The linearized metric is a per-user scalar — exactly the form that CUPED requires. We can combine the two techniques for maximum variance reduction:

1. **Linearize** the ratio metric: $L_i = x_i - \kappa \cdot y_i$
2. **Apply CUPED** using the pre-period linearized metric as the covariate

This is the state-of-the-art approach used by major tech companies.

In [ ]:
np.random.seed(42)

# Generate data with pre-period and pilot period
n_users = 4000
n_half = n_users // 2

# User-level characteristics (persistent across periods)
user_search_freq = np.random.lognormal(mean=1.5, sigma=0.8, size=n_users)
user_conv_rate = np.random.beta(2, 10, size=n_users)

def generate_period_data(n_users, user_search_freq, user_conv_rate, 
                         treatment_effect=0.0, treatment_mask=None):
    """Generate session-level data for one period."""
    user_sums = np.zeros(n_users)
    user_counts = np.zeros(n_users)
    
    for i in range(n_users):
        n_sessions = max(1, np.random.poisson(user_search_freq[i]))
        user_counts[i] = n_sessions
        
        conv_rate = user_conv_rate[i]
        if treatment_mask is not None and treatment_mask[i]:
            conv_rate *= (1 + treatment_effect)
        
        purchases = np.random.binomial(1, min(conv_rate, 1.0), n_sessions)
        revenues = purchases * np.random.lognormal(3, 1, n_sessions)
        user_sums[i] = revenues.sum()
    
    return user_sums, user_counts

# Pre-period (no treatment effect)
pre_sums, pre_counts = generate_period_data(n_users, user_search_freq, user_conv_rate)

# Pilot period (with treatment effect)
treatment_mask = np.zeros(n_users, dtype=bool)
treatment_mask[n_half:] = True
pilot_sums, pilot_counts = generate_period_data(
    n_users, user_search_freq, user_conv_rate,
    treatment_effect=0.06, treatment_mask=treatment_mask
)

# Compute kappa from control in pilot period
kappa_pilot = pilot_sums[:n_half].sum() / pilot_counts[:n_half].sum()
kappa_pre = pre_sums.sum() / pre_counts.sum()

# Linearize both periods
L_pilot = pilot_sums - kappa_pilot * pilot_counts
L_pre = pre_sums - kappa_pre * pre_counts

# Apply CUPED on the linearized metric
theta = np.cov(L_pilot, L_pre)[0, 1] / np.var(L_pre, ddof=1)
L_cuped = L_pilot - theta * (L_pre - L_pre.mean())

# Compare all approaches
print("=" * 70)
print("COMBINED LINEARIZATION + CUPED")
print("=" * 70)

methods_results = {
    'User-avg ratio (pilot)': (pilot_sums / pilot_counts, None),
    'Linearized (pilot)': (L_pilot, None),
    'Linearized + CUPED': (L_cuped, None),
}

print(f"\n{'Method':<30} {'Variance':>10} {'p-value':>10} {'Var Reduction':>14}")
print("-" * 70)

base_var = None
for name, (metric, _) in methods_results.items():
    ctrl = metric[:n_half]
    treat = metric[n_half:]
    var = (np.var(ctrl, ddof=1) + np.var(treat, ddof=1)) / 2
    _, p = stats.ttest_ind(treat, ctrl)
    
    if base_var is None:
        base_var = var
        reduction_str = '--'
    else:
        reduction_str = f"{(1 - var/base_var)*100:.1f}%"
    
    print(f"{name:<30} {var:>10.1f} {p:>10.4f} {reduction_str:>14}")

# Correlation between pre and pilot linearized metrics
rho_linear = np.corrcoef(L_pre, L_pilot)[0, 1]
print(f"\nCorrelation between pre/pilot linearized metrics: {rho_linear:.3f}")
print(f"Theoretical CUPED reduction on linearized metric: {rho_linear**2*100:.1f}%")

---

## 8. The Bucket Method: Handling Temporal Dependencies

### When User-Level Isn't Enough

Sometimes sessions within a user are temporally correlated (e.g., a user browsing intensely on one day). The **bucket method** aggregates data into time buckets (e.g., daily) to handle this:

1. For each day, compute the ratio metric: $R_d = \sum_i x_{i,d} / \sum_i y_{i,d}$
2. Treat daily ratios as observations in a t-test
3. This accounts for day-to-day correlation in user behavior

**Trade-off**: Fewer observations (number of days vs number of users), but correctly handles temporal dependence.

In [ ]:
# Bucket method implementation
def bucket_test(df, value_name, user_id_name, date_name, 
                control_users, treatment_users):
    """
    Bucket method: compute ratio per day, then t-test on daily ratios.
    """
    df_ctrl = df[df[user_id_name].isin(control_users)]
    df_treat = df[df[user_id_name].isin(treatment_users)]
    
    # Daily ratios for control
    ctrl_daily = df_ctrl.groupby(date_name).agg(
        total_value=(value_name, 'sum'),
        total_sessions=(value_name, 'count')
    )
    ctrl_daily['ratio'] = ctrl_daily['total_value'] / ctrl_daily['total_sessions']
    
    # Daily ratios for treatment
    treat_daily = df_treat.groupby(date_name).agg(
        total_value=(value_name, 'sum'),
        total_sessions=(value_name, 'count')
    )
    treat_daily['ratio'] = treat_daily['total_value'] / treat_daily['total_sessions']
    
    # Paired t-test (same days)
    _, p_value = stats.ttest_rel(treat_daily['ratio'].values, ctrl_daily['ratio'].values)
    
    return p_value, ctrl_daily['ratio'].values, treat_daily['ratio'].values


p_bucket, ctrl_daily_ratios, treat_daily_ratios = bucket_test(
    df_search, 'search_revenue', 'user_id', 'date',
    control_users, treatment_users
)

print(f"Bucket Method (paired t-test on daily ratios):")
print(f"  Control daily mean:   ${ctrl_daily_ratios.mean():.4f}")
print(f"  Treatment daily mean: ${treat_daily_ratios.mean():.4f}")
print(f"  p-value:              {p_bucket:.4f}")
print(f"  Number of buckets:    {len(ctrl_daily_ratios)} (days)")

# Visualize daily ratios
days = range(len(ctrl_daily_ratios))
plt.plot(days, ctrl_daily_ratios, 'o-', label='Control', linewidth=2)
plt.plot(days, treat_daily_ratios, 's-', label='Treatment', linewidth=2)
plt.xlabel('Day of Experiment')
plt.ylabel('Revenue per Search Session ($)')
plt.title('Bucket Method: Daily Ratio Metric')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print(f"\nNote: With only {len(ctrl_daily_ratios)} days, the bucket method has low power.")
print(f"It's most useful for detecting large effects or when user-level independence is questionable.")

---

## 9. Comprehensive Calibration Check

Let's run a definitive A/A calibration test comparing the linearized metric against the user-average approach, focusing on the scenario where session counts vary widely (the realistic case).

In [ ]:
def run_calibration_by_heterogeneity(session_heterogeneity_levels, n_simulations=3000):
    """
    Test calibration as we increase heterogeneity in session counts.
    Higher heterogeneity = more variable session counts = bigger problem for user-avg.
    """
    results = {'heterogeneity': [], 'method': [], 'fpr': []}
    n_users = 2000
    n_half = n_users // 2
    
    for p_param in session_heterogeneity_levels:
        pvals_ua = []
        pvals_lin = []
        
        for _ in range(n_simulations):
            # Higher p_param -> less heterogeneity in NB distribution
            sessions = np.random.negative_binomial(n=2, p=p_param, size=n_users) + 1
            
            # Generate revenue (no treatment effect)
            user_sums = np.zeros(n_users)
            for i in range(n_users):
                user_sums[i] = np.random.exponential(5.0, sessions[i]).sum()
            
            # Split
            sums_a, sums_b = user_sums[:n_half], user_sums[n_half:]
            counts_a, counts_b = sessions[:n_half], sessions[n_half:]
            
            # User-average
            _, p_ua = stats.ttest_ind(sums_a / counts_a, sums_b / counts_b)
            pvals_ua.append(p_ua)
            
            # Linearization
            kappa = user_sums.sum() / sessions.sum()
            _, p_lin = stats.ttest_ind(
                sums_a - kappa * counts_a,
                sums_b - kappa * counts_b
            )
            pvals_lin.append(p_lin)
        
        cv_sessions = np.std(sessions) / np.mean(sessions)  # coefficient of variation
        results['heterogeneity'].append(cv_sessions)
        results['method'].append('User-Average')
        results['fpr'].append(np.mean(np.array(pvals_ua) < 0.05))
        
        results['heterogeneity'].append(cv_sessions)
        results['method'].append('Linearization')
        results['fpr'].append(np.mean(np.array(pvals_lin) < 0.05))
    
    return pd.DataFrame(results)


# Run across different heterogeneity levels
p_params = [0.5, 0.3, 0.2, 0.15, 0.10, 0.07, 0.05]
print("Running calibration test across session count heterogeneity levels...")
calibration_df = run_calibration_by_heterogeneity(p_params, n_simulations=3000)

# Plot
for method in ['User-Average', 'Linearization']:
    subset = calibration_df[calibration_df['method'] == method]
    plt.plot(subset['heterogeneity'], subset['fpr'], 'o-', label=method, linewidth=2)

plt.axhline(0.05, color='red', linestyle='--', linewidth=2, label='Target FPR = 0.05')
plt.fill_between([0, 2.5], 0.04, 0.06, alpha=0.1, color='red', label='Acceptable range')
plt.xlabel('Session Count Heterogeneity (CV of sessions/user)')
plt.ylabel('False Positive Rate')
plt.title('Calibration vs Session Count Heterogeneity')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print("\nAs session count heterogeneity increases:")
print("- User-Average may drift from the target FPR")
print("- Linearization stays correctly calibrated")

---

## 10. The Delta Method (For Reference)

The delta method provides an analytical formula for the variance of a ratio. While linearization is more practical, understanding the delta method clarifies *why* linearization works.

:::{admonition} Delta method variance formula (click to expand)
:class: dropdown

For a ratio $R = \bar{X}/\bar{Y}$ where observations are at the user level ($X_i$ = total revenue for user $i$, $Y_i$ = session count for user $i$):

$$
\text{Var}(R) \approx \frac{1}{n} \cdot \frac{1}{\mu_Y^2} \left[ \sigma_X^2 - 2R\,\text{Cov}(X,Y) + R^2 \sigma_Y^2 \right]
$$

This is exactly the variance of $(X_i - R \cdot Y_i)$ divided by $n \mu_Y^2$.

Notice: $X_i - R \cdot Y_i$ is precisely the linearized metric! So the delta method and linearization give **identical** variance estimates (and therefore identical p-values) asymptotically.
:::

In [ ]:
def delta_method_variance(sums, counts, n_users):
    """
    Compute variance of ratio using delta method.
    sums: per-user sum of numerator (revenue)
    counts: per-user count of denominator (sessions)
    """
    mu_x = np.mean(sums)
    mu_y = np.mean(counts)
    R = mu_x / mu_y  # ratio estimate
    
    var_x = np.var(sums, ddof=1)
    var_y = np.var(counts, ddof=1)
    cov_xy = np.cov(sums, counts, ddof=1)[0, 1]
    
    var_ratio = (1.0 / (n_users * mu_y**2)) * (var_x - 2 * R * cov_xy + R**2 * var_y)
    return var_ratio


def linearization_variance(sums, counts, kappa, n_users):
    """Compute variance of linearized metric, scaled for ratio comparison."""
    L = sums - kappa * counts
    mu_y = np.mean(counts)
    var_L = np.var(L, ddof=1)
    # Scale by 1/(n * mu_y^2) to compare with delta method
    return var_L / (n_users * mu_y**2)


# Compare delta method and linearization variances
np.random.seed(42)
n = 5000
sessions = np.random.negative_binomial(n=2, p=0.1, size=n) + 1
revenues = np.array([np.random.exponential(5, s).sum() for s in sessions])
kappa = revenues.sum() / sessions.sum()

var_delta = delta_method_variance(revenues, sessions, n)
var_linear = linearization_variance(revenues, sessions, kappa, n)

print(f"Variance estimates for the ratio metric:")
print(f"  Delta method:    {var_delta:.6f}")
print(f"  Linearization:   {var_linear:.6f}")
print(f"  Ratio:           {var_linear/var_delta:.4f}")
print(f"\nThey are asymptotically equivalent (ratio approaches 1 as n grows).")

---

## 11. Full Pipeline: Linearization + CUPED on QuickCart Data

Let's put it all together with a complete end-to-end example.

In [ ]:
np.random.seed(42)
n_users = 6000
n_half = n_users // 2

# Persistent user traits
user_activity = np.random.lognormal(mean=1.5, sigma=1.0, size=n_users)
user_spending = np.random.beta(2, 8, size=n_users) * 50  # avg order value scale

def simulate_period(user_activity, user_spending, n_users, 
                    treatment_effect=0.0, treatment_mask=None):
    """Simulate one period of search sessions."""
    sums = np.zeros(n_users)
    counts = np.zeros(n_users)
    
    for i in range(n_users):
        n_sessions = max(1, np.random.poisson(user_activity[i]))
        counts[i] = n_sessions
        
        for s in range(n_sessions):
            # Base conversion rate
            conv = 0.15
            if treatment_mask is not None and treatment_mask[i]:
                conv *= (1 + treatment_effect)
            
            if np.random.random() < conv:
                sums[i] += np.random.exponential(user_spending[i])
    
    return sums, counts

# Pre-period (no treatment)
pre_sums, pre_counts = simulate_period(user_activity, user_spending, n_users)

# Pilot period (with 5% treatment effect)
treatment_mask = np.zeros(n_users, dtype=bool)
treatment_mask[n_half:] = True
pilot_sums, pilot_counts = simulate_period(
    user_activity, user_spending, n_users,
    treatment_effect=0.05, treatment_mask=treatment_mask
)

# Step 1: Compute kappa (from control group in pilot)
kappa = pilot_sums[:n_half].sum() / pilot_counts[:n_half].sum()
kappa_pre = pre_sums.sum() / pre_counts.sum()

# Step 2: Linearize
L_pilot = pilot_sums - kappa * pilot_counts
L_pre = pre_sums - kappa_pre * pre_counts

# Step 3: Apply CUPED on linearized metrics
theta = np.cov(L_pilot, L_pre)[0, 1] / np.var(L_pre, ddof=1)
L_cuped = L_pilot - theta * (L_pre - L_pre.mean())

# Compare all methods
print("=" * 75)
print("FULL PIPELINE COMPARISON: Revenue per Search Session")
print("True treatment effect: 5% lift in conversion rate")
print("=" * 75)

# Raw ratio
ratio_ctrl = pilot_sums[:n_half] / pilot_counts[:n_half]
ratio_treat = pilot_sums[n_half:] / pilot_counts[n_half:]

approaches = [
    ('1. User-avg ratio', ratio_ctrl, ratio_treat),
    ('2. Linearized', L_pilot[:n_half], L_pilot[n_half:]),
    ('3. Linearized + CUPED', L_cuped[:n_half], L_cuped[n_half:]),
]

print(f"\n{'Method':<25} {'Ctrl Mean':>10} {'Treat Mean':>11} {'Var(pooled)':>12} {'p-value':>10} {'Sig?':>5}")
print("-" * 75)

for name, ctrl, treat in approaches:
    var_pooled = (np.var(ctrl, ddof=1) + np.var(treat, ddof=1)) / 2
    _, p = stats.ttest_ind(treat, ctrl)
    sig = 'Yes' if p < 0.05 else 'No'
    print(f"{name:<25} {ctrl.mean():>10.2f} {treat.mean():>11.2f} {var_pooled:>12.1f} {p:>10.4f} {sig:>5}")

# Variance reduction summary
var_base = (np.var(L_pilot[:n_half], ddof=1) + np.var(L_pilot[n_half:], ddof=1)) / 2
var_cuped = (np.var(L_cuped[:n_half], ddof=1) + np.var(L_cuped[n_half:], ddof=1)) / 2
rho = np.corrcoef(L_pre, L_pilot)[0, 1]

print(f"\nPre/Pilot linearized correlation: {rho:.3f}")
print(f"CUPED variance reduction: {(1 - var_cuped/var_base)*100:.1f}%")
print(f"Theoretical reduction (rho^2): {rho**2*100:.1f}%")

---

## 12. Diagnostic: When Does Linearization Matter Most?

Let's visualize when the user-average approach breaks down most severely.

In [ ]:
# Show the relationship between session count and ratio estimate precision
np.random.seed(42)
n = 3000
true_ratio = 5.0

sessions = np.random.negative_binomial(n=2, p=0.1, size=n) + 1
revenues = np.array([np.random.exponential(true_ratio, s).sum() for s in sessions])
user_ratios = revenues / sessions

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: ratio vs session count
axes[0].scatter(sessions, user_ratios, alpha=0.3, s=10)
axes[0].axhline(true_ratio, color='red', linestyle='--', linewidth=2, label=f'True ratio = {true_ratio}')
axes[0].set_xlabel('Number of Sessions per User')
axes[0].set_ylabel('User-Level Ratio (revenue/sessions)')
axes[0].set_title('User Ratios vs Session Count')
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[0].set_xlim([0, 80])
axes[0].set_ylim([0, 40])

# Right: variance of ratio estimates by session count bucket
session_buckets = pd.cut(sessions, bins=[0, 3, 7, 15, 30, 200], labels=['1-3', '4-7', '8-15', '16-30', '31+'])
bucket_vars = pd.DataFrame({'sessions_bucket': session_buckets, 'ratio': user_ratios})
bucket_stats = bucket_vars.groupby('sessions_bucket', observed=True)['ratio'].agg(['var', 'count'])

axes[1].bar(range(len(bucket_stats)), bucket_stats['var'], color='steelblue', alpha=0.7)
axes[1].set_xticks(range(len(bucket_stats)))
axes[1].set_xticklabels(bucket_stats.index)
axes[1].set_xlabel('Sessions per User (bucket)')
axes[1].set_ylabel('Variance of User-Level Ratio')
axes[1].set_title('Ratio Variance by Session Count\n(fewer sessions = noisier ratio)')
axes[1].grid(alpha=0.3, axis='y')

# Add count labels
for i, (_, row) in enumerate(bucket_stats.iterrows()):
    axes[1].text(i, row['var'] + 0.5, f'n={int(row["count"])}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

print("Key insight: Users with few sessions have wildly imprecise ratio estimates.")
print("In the user-average approach, these noisy estimates get equal weight,")
print("reducing power and potentially introducing bias.")
print("\nLinearization naturally downweights low-session users because their")
print("linearized metric L_i = x_i - kappa*y_i is small in magnitude.")

---

## Summary

| Concept | Key Takeaway |
|---------|-------------|
| Ratio metric problem | Users have different session counts; naive averaging is biased |
| User-average method | Simple but overweights low-session users; can have wrong Type I error |
| Delta method | Analytically correct variance; equivalent to linearization asymptotically |
| Linearization formula | $L_i = x_i - \kappa \cdot y_i$ where $\kappa = \sum x / \sum y$ |
| Why it works | First-order Taylor expansion of the ratio; converts ratio to sum |
| Kappa estimation | Use control group or pooled data; must be computed BEFORE looking at results |
| Linearization + CUPED | Linearize first, then apply CUPED on the linearized metric |
| Bucket method | Aggregate by time bucket; handles temporal correlation; low power |
| When linearization matters most | High heterogeneity in session counts across users |
| Validation | Always run A/A test to confirm uniform p-values |

### What's Next

In **Week 7**, we'll address the multiple testing problem — what happens when you test many metrics simultaneously or peek at results during the experiment. We'll cover Bonferroni, BH-FDR, and their interactions with the variance reduction techniques from Weeks 5-6.